# Notebook 13 — Preventing Data Leakage
### Sprint 5 | Data Cleaning & Preprocessing for AI/ML Engineers

**This topic is mandatory because data leakage can produce misleadingly high model
performance** — every leakage type below is demonstrated with a real, numerically
measured "Incorrect Workflow → Correct Workflow" comparison, not just described.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = pd.read_csv("telco_churn.csv")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
print(f"Dataset ready: {df.shape[0]:,} rows")


Dataset ready: 7,043 rows


---
## 1. What is Data Leakage?

### Understand
Data leakage happens when information that would NOT actually be available at real
prediction time leaks into model training — causing performance metrics that look great
in development but collapse (or simply can't be reproduced) in real deployment.


---
## 2. Target Leakage

### Understand
A feature that's a direct byproduct of the target itself — it only exists *because* the
outcome already happened.

### Incorrect Workflow → Correct Workflow


In [2]:
df_leak = df.copy()
df_leak['had_retention_call'] = np.where(df_leak['Churn'] == 'Yes', 1, 0)   # only logged AFTER a customer churned

X_leaky = df_leak[['tenure', 'had_retention_call']]
X_clean = df_leak[['tenure']]
y = (df_leak['Churn'] == 'Yes').astype(int)

for name, X_variant in [('INCORRECT (target leakage)', X_leaky), ('CORRECT (no leakage)', X_clean)]:
    X_tr, X_te, y_tr, y_te = train_test_split(X_variant, y, test_size=0.2, stratify=y, random_state=42)
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    print(f"{name}: test accuracy = {model.score(X_te, y_te):.4f}")


INCORRECT (target leakage): test accuracy = 1.0000
CORRECT (no leakage): test accuracy = 0.7324


**Finding:** The leaky version scores near-perfect accuracy — not because it
learned anything real, but because `had_retention_call` IS effectively the answer,
recorded after the fact. In real deployment, this field would never exist for a
not-yet-churned customer, so the model would be useless the moment it's actually needed.


---
## 3. Train-Test Contamination

### Understand
Any preprocessing statistic (scaler mean, imputer fill value, encoder categories)
computed using BOTH train and test data before splitting — even without an obviously
leaky feature like Topic 2's example.

### Incorrect Workflow → Correct Workflow


In [3]:
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# INCORRECT: fit scaler on the FULL dataset before splitting
scaler_wrong = StandardScaler().fit(df[numeric_cols])
X_all_scaled_wrong = pd.DataFrame(scaler_wrong.transform(df[numeric_cols]), columns=numeric_cols, index=df.index)
X_tr_wrong, X_te_wrong, y_tr, y_te = train_test_split(X_all_scaled_wrong, y, test_size=0.2, stratify=y, random_state=42)
model_wrong = LogisticRegression(max_iter=1000).fit(X_tr_wrong, y_tr)

# CORRECT: split FIRST, fit scaler on train only
X_tr_raw, X_te_raw, y_tr2, y_te2 = train_test_split(df[numeric_cols], y, test_size=0.2, stratify=y, random_state=42)
scaler_right = StandardScaler().fit(X_tr_raw)
X_tr_right = scaler_right.transform(X_tr_raw)
X_te_right = scaler_right.transform(X_te_raw)
model_right = LogisticRegression(max_iter=1000).fit(X_tr_right, y_tr2)

print(f"INCORRECT (scaler fit on full data) test accuracy: {model_wrong.score(X_te_wrong, y_te):.4f}")
print(f"CORRECT (scaler fit on train only)  test accuracy: {model_right.score(X_te_right, y_te2):.4f}")
print(f"\nScaler mean, INCORRECT (uses all 7,043 rows): {scaler_wrong.mean_.round(2)}")
print(f"Scaler mean, CORRECT (uses only {len(X_tr_raw):,} training rows): {scaler_right.mean_.round(2)}")


INCORRECT (scaler fit on full data) test accuracy: 0.7722
CORRECT (scaler fit on train only)  test accuracy: 0.7722

Scaler mean, INCORRECT (uses all 7,043 rows): [  32.37   64.76 2279.73]
Scaler mean, CORRECT (uses only 5,634 training rows): [  32.49   64.93 2299.33]


**Finding:** The accuracy difference is small here (this dataset's train/test
distributions are similar since the split is random and large), but the scaler's learned
mean is measurably different — confirming real contamination occurred, even though its
practical impact on this particular model is modest. On a smaller dataset, or one with
more distribution shift between folds, this contamination could meaningfully inflate
reported performance.


---
## 4. Feature Leakage

### Understand
A feature that's technically available at prediction time, but only because it was
DERIVED using information that itself depends on the target or on future information —
subtler than direct target leakage.

### Incorrect Workflow → Correct Workflow


In [4]:
# INCORRECT: engineering a feature using statistics computed from the FULL dataset (train+test)
# including the target-correlated grouping
df_feat = df.copy()
df_feat['Churn_numeric'] = (df_feat['Churn'] == 'Yes').astype(int)

# leaky: average churn rate per Contract type, computed using ALL data (including test rows)
leaky_target_encoding = df_feat.groupby('Contract')['Churn_numeric'].transform('mean')

# correct: the same idea, but computed using ONLY the training fold, then mapped onto test
train_idx, test_idx = train_test_split(df_feat.index, test_size=0.2, stratify=y, random_state=42)
contract_means_train_only = df_feat.loc[train_idx].groupby('Contract')['Churn_numeric'].mean()
correct_target_encoding = df_feat['Contract'].map(contract_means_train_only)

print("Leaky encoding uses test rows' OWN churn outcomes to help predict those same rows — circular.")
print(f"Contract='Month-to-month' leaky encoding (includes test rows): {leaky_target_encoding[df_feat['Contract']=='Month-to-month'].iloc[0]:.4f}")
print(f"Contract='Month-to-month' correct encoding (train rows only) : {contract_means_train_only['Month-to-month']:.4f}")


Leaky encoding uses test rows' OWN churn outcomes to help predict those same rows — circular.
Contract='Month-to-month' leaky encoding (includes test rows): 0.4271
Contract='Month-to-month' correct encoding (train rows only) : 0.4275


**Finding:** The two values are close but NOT identical — the leaky version has
been computed partly using the very rows it will later be evaluated on, creating a
subtle circularity a naive review might miss entirely, since `Contract` itself looks like
a perfectly legitimate feature.


---
## 5. Preprocessing Leakage

### Understand
A specific case of train-test contamination: fitting ANY preprocessing (imputer,
encoder, feature selector, resampler) on the full dataset — this generalizes Topic 3's
scaler example to every preprocessing step in this sprint.

### Incorrect Workflow → Correct Workflow


In [5]:
from sklearn.impute import SimpleImputer

demo_col = df[['MonthlyCharges']].copy()
demo_col.iloc[0:5] = np.nan   # inject synthetic missingness for this illustration

# INCORRECT: impute using the full column's mean (train+test combined)
imputer_wrong = SimpleImputer(strategy='mean').fit(demo_col)
print(f"INCORRECT — imputer fit on FULL data, mean = {imputer_wrong.statistics_[0]:.2f}")

# CORRECT: split first, impute using only the training portion's mean
train_part, test_part = train_test_split(demo_col, test_size=0.2, random_state=42)
imputer_right = SimpleImputer(strategy='mean').fit(train_part)
print(f"CORRECT   — imputer fit on TRAIN ONLY, mean = {imputer_right.statistics_[0]:.2f}")


INCORRECT — imputer fit on FULL data, mean = 64.77
CORRECT   — imputer fit on TRAIN ONLY, mean = 64.88


**Finding:** Same underlying principle as Topic 3, now shown for imputation
specifically — the fix is identical every time: split first, fit second, on every single
preprocessing object in the pipeline (Notebook 14 formalizes this with `Pipeline`).


---
## 6. Temporal Leakage

### Understand
Using information from AFTER the prediction point in time — e.g., a feature that
reflects a customer's status next month, used to predict this month's outcome.

### Demonstrate


In [6]:
print("This dataset has no timestamps (confirmed Sprint 4/5), so temporal leakage")
print("cannot be directly demonstrated on it. The general pattern, illustrated:")
print("  INCORRECT: predicting January churn using a 'total 2024 charges' feature")
print("             that includes charges from March-December — not yet known in January.")
print("  CORRECT  : only use information genuinely available AT the prediction time,")
print("             e.g., 'charges accumulated through January only'.")


This dataset has no timestamps (confirmed Sprint 4/5), so temporal leakage
cannot be directly demonstrated on it. The general pattern, illustrated:
  INCORRECT: predicting January churn using a 'total 2024 charges' feature
             that includes charges from March-December — not yet known in January.
  CORRECT  : only use information genuinely available AT the prediction time,
             e.g., 'charges accumulated through January only'.


**Not directly demonstrable on this dataset** (no dates) — documented honestly,
with the general principle explained rather than forced onto data that doesn't fit it.


---
## 7. Examples of Data Leakage — Consolidated

### Implement


In [7]:
leakage_examples = pd.DataFrame({
    'Type': ['Target Leakage', 'Train-Test Contamination', 'Feature Leakage', 'Preprocessing Leakage', 'Temporal Leakage'],
    'Example in this notebook': [
        "had_retention_call (only exists after churn)",
        "StandardScaler fit on full dataset before split",
        "Contract's churn-rate encoding computed on train+test",
        "SimpleImputer mean computed on full dataset",
        "Not directly demonstrable (no dates in this dataset)"
    ],
    'Demonstrated Numerically?': ['Yes', 'Yes', 'Yes', 'Yes', 'Explained conceptually only']
})
print(leakage_examples.to_string(index=False))


                    Type                              Example in this notebook   Demonstrated Numerically?
          Target Leakage          had_retention_call (only exists after churn)                         Yes
Train-Test Contamination       StandardScaler fit on full dataset before split                         Yes
         Feature Leakage Contract's churn-rate encoding computed on train+test                         Yes
   Preprocessing Leakage           SimpleImputer mean computed on full dataset                         Yes
        Temporal Leakage  Not directly demonstrable (no dates in this dataset) Explained conceptually only


---
## 8. How to Detect Leakage & 9. How to Prevent Leakage

### Understand
**Detection signals:** suspiciously high accuracy (especially near 100%), a feature with
an unreasonably strong correlation with the target, or performance that collapses when
moved to genuinely new data.
**Prevention:** always split before fitting anything; use `Pipeline`/`ColumnTransformer`
(Notebook 14) so fitting is structurally forced to happen only on training folds; audit
every feature by asking "would this value actually be known at real prediction time?"

### Implement — A Simple Leakage Detector


In [8]:
def check_suspiciously_high_correlation(data, target_col, threshold=0.9):
    correlations = data.corr()[target_col].drop(target_col).abs()
    suspicious = correlations[correlations > threshold]
    return suspicious

df_leak['Churn_numeric'] = (df_leak['Churn'] == 'Yes').astype(int)
check_df = df_leak[['tenure', 'had_retention_call', 'Churn_numeric']]
suspicious = check_suspiciously_high_correlation(check_df, 'Churn_numeric')
print(f"Features with suspiciously high (>0.9) correlation with the target: \n{suspicious if len(suspicious) else 'None'}")


Features with suspiciously high (>0.9) correlation with the target: 
had_retention_call    1.0
Name: Churn_numeric, dtype: float64


**Finding:** The detector correctly flags `had_retention_call` — a fast, automated
first-pass check worth running on any new feature before trusting it.


---
## Summary

| Leakage Type | Demonstrated? | Fix |
|---|---|---|
| Target Leakage | Yes — near-perfect accuracy from a post-outcome feature | Remove any feature only knowable after the target |
| Train-Test Contamination | Yes — scaler mean differs measurably | Split before fitting anything |
| Feature Leakage | Yes — target-encoded value differs when computed correctly | Compute derived stats using train fold only |
| Preprocessing Leakage | Yes — imputer mean differs | Same split-first discipline for every preprocessing step |
| Temporal Leakage | Explained conceptually (no dates in this dataset) | Never use future information relative to the prediction point |

**Why this matters, concretely:** every "Incorrect" workflow above still ran without
error and produced a plausible-looking result — leakage doesn't announce itself with a
crash, it quietly inflates performance. This is exactly why it's described as producing
*misleadingly* high performance, not just *wrong* performance.

**Next notebook:** `14_Preprocessing_Pipeline.ipynb` — using scikit-learn's `Pipeline`
and `ColumnTransformer` to make the "split first, fit second" discipline structurally
enforced rather than something to remember manually.
